# 教程: 时间步长的影响与选择

## 目的
在任何数值模拟中，时间步长（`dt`）的选择都是一个至关重要的决定。它直接影响到模型的**稳定性（Stability）**、**准确性（Accuracy）**和**计算效率（Efficiency）**。
- **过大的 `dt`**: 可能导致数值解发散（即“模型崩溃”），或者得到一个与物理现实严重不符的、不准确的结果。
- **过小的 `dt`**: 虽然通常能保证稳定性和准确性，但会大大增加计算时间，降低效率。

本教程旨在通过实验，直观地展示`dt`的选择对一维水力模型结果的影响。

此笔记本将：
1.  建立一个标准的一维水力模型。
2.  使用一系列不同的时间步长（从非常小到可能过大）来运行同一个模拟场景。
3.  比较不同`dt`下的模拟结果（出口过程线）和计算耗时。
4.  讨论如何在准确性和效率之间进行权衡，以选择一个合适的`dt`。

## 1. 环境设置和模型准备

我们使用`preissmann_model`作为测试对象，因为它是一个基于物理方程的动态模型，对时间步长的选择相对敏感。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sys
import os
import time

# 将项目根目录添加到Python路径
sys.path.insert(0, os.path.abspath(os.path.join(os.path.dirname('__file__'), '..')))

from preissmann_model.model import HydraulicModel
from preissmann_model.reach import RiverReach
from preissmann_model.cross_section import RectangularCrossSection

# --- 定义一个辅助函数来创建和运行模型 ---
def create_and_run_model(dt_value, q_inflow):
    reach_geom = RiverReach(
        cross_sections=[RectangularCrossSection(width=20) for _ in range(11)],
        lengths=np.full(10, 200.0),
        slope=0.001, manning_n=0.03
    )
    model = HydraulicModel(name=f"model_dt_{dt_value}", reach=reach_geom, dt=dt_value, downstream_level=2.0)
    model.Q[:] = q_inflow[0]
    model.Z[:] = 2.5
    
    num_steps = int(total_sim_time / dt_value)
    q_inflow_resampled = np.interp(np.linspace(0, total_sim_time, num_steps), np.linspace(0, total_sim_time, len(q_inflow)), q_inflow)
    
    outflows = []
    start_time = time.time()
    try:
        for i in range(num_steps):
            model.step({'Q_inflow': q_inflow_resampled[i]}, dt_value)
            outflows.append(model.get_outflow())
    except np.linalg.LinAlgError:
        print(f"dt = {dt_value}s 时模型出现数值不稳定!")
        return None, None, time.time() - start_time
        
    end_time = time.time()
    run_time = end_time - start_time
    timesteps = np.arange(num_steps) * dt_value
    return timesteps, np.array(outflows), run_time

# --- 定义共享的输入 ---
total_sim_time = 3600 # 总模拟时长 (秒)
base_q_inflow = np.array([10]*5 + [20, 40, 60, 40, 20] + [10]*5)

## 2. 运行不同`dt`的模拟

我们选择一组`dt`值，从10秒到120秒，并为每个值运行一次模拟。

In [ ]:
dt_values = [10, 30, 60, 120]
results = {}

for dt in dt_values:
    print(f"--- 正在运行 dt = {dt}s 的模拟 ---")
    timesteps, outflows, run_time = create_and_run_model(dt, base_q_inflow)
    results[dt] = {
        'timesteps': timesteps,
        'outflows': outflows,
        'run_time': run_time
    }
    if outflows is not None:
        print(f"  ...运行成功，耗时 {run_time:.4f}s")

## 3. 结果对比

### 3.1 准确性对比 (出口过程线)

In [ ]:
plt.figure(figsize=(12, 7))

for dt, res in results.items():
    if res['outflows'] is not None:
        plt.plot(res['timesteps'], res['outflows'], '-o', markersize=4, label=f'dt = {dt}s')

# 绘制原始入流作为参考
plt.plot(np.linspace(0, total_sim_time, len(base_q_inflow)), base_q_inflow, 'k--', label='Inflow Hydrograph')

plt.title('Impact of Time Step (dt) on Accuracy')
plt.xlabel('Time (s)')
plt.ylabel('Discharge (m³/s)')
plt.legend()
plt.grid(True)
plt.show()

从上图可以看出，当`dt`从10秒增加到60秒时，结果的差异很小，说明在这个范围内，模型的结果是比较收敛的。然而，当`dt`增大到120秒时，虽然模型没有崩溃，但其计算出的洪峰明显低于其他结果，这被称为**数值耗散（Numerical Dissipation）**，是一种由过大时间步长引起的精度损失。

### 3.2 效率对比 (计算时间)

In [ ]:
run_times = {f"dt={dt}": res['run_time'] for dt, res in results.items() if res['run_time'] is not None}

plt.figure(figsize=(8, 5))
plt.bar(run_times.keys(), run_times.values(), color='c')
plt.title('Impact of Time Step (dt) on Computation Time')
plt.ylabel('Wall Clock Time (s)')
plt.show()

计算时间的结果则完全符合预期：`dt`越小，所需的总步数就越多，因此总计算时间也越长。

## 4. 结论
选择时间步长`dt`是一个在**准确性**和**效率**之间的权衡过程。
- 为了获得最准确的结果，我们应该使用足够小的`dt`，使得结果不再随`dt`的进一步减小而发生显著变化。在这个例子中，`dt=10s`或`dt=30s`似乎都是不错的选择。
- 如果计算效率是首要考虑的，我们可以选择一个更大的`dt`（如`dt=60s`），只要我们能够接受它所带来的微小精度损失。
- 我们必须避免使用过大的`dt`（如`dt=120s`），因为它会导致结果不准确，甚至可能导致模型不稳定而崩溃。

在实际应用中，进行类似的**时间步长敏感性分析**是确保模型结果可靠的重要步骤。